In [16]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
from scipy.io import mmread
from scipy.sparse import csr_matrix
from anndata import AnnData

# List of study folders
folders = [
    "Chen_2024",
    "Chow_2023",
    "Liu_2022",
    "Liu_2025",
    "Zheng_2021"
]

data_dir = "/Users/wsun/research/CAT/"
meta_dir = "/Users/wsun/research/CAT/data"

## Read in data

In [17]:
# Step 1: Read and collect gene names
gene_lists = {}
adata_dict = {}

for folder in folders:
    print(f"Reading {folder}...")

    # Read files
    matrix = mmread(f"{data_dir}{folder}/{folder}_CD8/matrix.mtx.gz").tocsr()
    genes = pd.read_csv(f"{data_dir}{folder}/{folder}_CD8/genes.tsv", header=None, sep="\t")[0]
    barcodes = pd.read_csv(f"{data_dir}{folder}/{folder}_CD8/barcodes.tsv", header=None)[0]

    # Store gene list for intersection
    gene_lists[folder] = genes

    # Create AnnData object
    adata = AnnData(X=matrix)
    adata.var_names = genes
    adata.obs_names = barcodes
    adata.obs["study"] = folder

    adata_dict[folder] = adata

Reading Chen_2024...
Reading Chow_2023...
Reading Liu_2022...
Reading Liu_2025...
Reading Zheng_2021...


In [18]:
for k, v in adata_dict.items():
    print(f"{k}: {v.shape}")

Chen_2024: (206336, 16323)
Chow_2023: (57516, 16323)
Liu_2022: (59609, 16323)
Liu_2025: (347702, 16323)
Zheng_2021: (70301, 16323)


In [19]:
# Step 2: Intersect genes
common_genes = set.intersection(*(set(g) for g in gene_lists.values()))
common_genes = sorted(list(common_genes))  # Ensure consistent order
print(f"Found {len(common_genes)} common genes.")


Found 16323 common genes.


In [20]:
# Step 3: Subset each AnnData to common genes
for k in adata_dict:
    adata_dict[k] = adata_dict[k][:, common_genes]

# Step 4: Concatenate into one AnnData object
adata_combined = adata_dict[folders[0]].concatenate(
    *[adata_dict[f] for f in folders[1:]],
    batch_key="study",
    batch_categories=folders
)

print(f"Combined AnnData shape: {adata_combined.shape}")
adata_combined

/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_24480/4214255207.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_combined = adata_dict[folders[0]].concatenate(


Combined AnnData shape: (741464, 16323)


AnnData object with n_obs × n_vars = 741464 × 16323
    obs: 'study'

In [21]:
sc.pp.calculate_qc_metrics(
    adata_combined,
    percent_top=None,      # you can set e.g., [50, 100, 200] if you want top-n genes metrics
    log1p=False,           # don't log-transform counts
    inplace=True           # store results directly in adata.obs / adata.var
)
print(adata_combined.obs.head())



                                          study  n_genes_by_counts  \
CRC01-N-I_AAAGATGGTCCGAGTC-Chen_2024  Chen_2024               1483   
CRC01-N-I_AAAGATGTCGCTAGCG-Chen_2024  Chen_2024               1700   
CRC01-N-I_AAAGCAAAGACTAGGC-Chen_2024  Chen_2024               1590   
CRC01-N-I_AAAGCAATCCTAGGGC-Chen_2024  Chen_2024               1159   
CRC01-N-I_AAAGTAGCAACGATCT-Chen_2024  Chen_2024               1339   

                                      total_counts  
CRC01-N-I_AAAGATGGTCCGAGTC-Chen_2024        3518.0  
CRC01-N-I_AAAGATGTCGCTAGCG-Chen_2024        4520.0  
CRC01-N-I_AAAGCAAAGACTAGGC-Chen_2024        4468.0  
CRC01-N-I_AAAGCAATCCTAGGGC-Chen_2024        2974.0  
CRC01-N-I_AAAGTAGCAACGATCT-Chen_2024        2951.0  


In [22]:
X = adata_combined.X
print(f"adata.X shape: {X.shape}")
print(X[100:110,100:105])

# Get the number of cells
n_cells = X.shape[0]

# Count how many cells express each gene (non-zero)
gene_expr_counts = (X > 0).sum(axis=0)
gene_expr_counts = np.asarray(gene_expr_counts).flatten()

# Compute fraction of expressing cells per gene
gene_expr_frac = gene_expr_counts / n_cells

# Convert to pandas Series for convenience
gene_frac_series = pd.Series(gene_expr_frac, index=adata_combined.var_names)

# Define thresholds
thresholds = [0.01, 0.02, 0.03, 0.05]

# Count how many genes pass each threshold
for t in thresholds:
    count = (gene_frac_series >= t).sum()
    print(f"Genes expressed in ≥{int(t * 100)}% of cells: {count}")


adata.X shape: (741464, 16323)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (10, 5)>
  Coords	Values
  (1, 1)	1.0
  (4, 1)	1.0
  (5, 2)	1.0
  (5, 1)	2.0
  (6, 1)	1.0
  (7, 1)	1.0
  (9, 1)	3.0
Genes expressed in ≥1% of cells: 9510
Genes expressed in ≥2% of cells: 8514
Genes expressed in ≥3% of cells: 7656
Genes expressed in ≥5% of cells: 6256


In [23]:
keep_genes = gene_expr_frac >= 0.05
genes_to_keep = adata_combined.var_names[keep_genes]

adata_combined = adata_combined[:, genes_to_keep].copy()
print(f"Filtered to {adata_combined.n_vars} genes expressed in ≥5% of cells.")

Filtered to 6256 genes expressed in ≥5% of cells.


In [24]:
file_path = os.path.join(meta_dir, "CD8_updated.tsv.gz")
cell_info = pd.read_csv(file_path, sep="\t", compression="gzip")

print(cell_info.shape)
print(cell_info.head())

(371010, 42)
        TRB_cdr3 TRB_v_gene         TRA_cdr3    TRA_v_gene  \
0  CAAADRGQDTQYF     TRBV19    CAADNQAGTALIF      TRAV13-1   
1  CAAADRGQDTQYF     TRBV19    CAADNQAGTALIF      TRAV13-1   
2  CAAADRGQDTQYF     TRBV19    CAVPQEGNNRLAF       TRAV8-6   
3  CAAAETSNQPQHF      TRBV2   CARPLGGSNYKLTF        TRAV16   
4   CAAAGIYNEQFF    TRBV6-2  CAYRRSASGTYKYIF  TRAV38-2/DV8   

                                               clone  \
0        TRAV13-1_CAADNQAGTALIF_TRBV19_CAAADRGQDTQYF   
1        TRAV13-1_CAADNQAGTALIF_TRBV19_CAAADRGQDTQYF   
2         TRAV8-6_CAVPQEGNNRLAF_TRBV19_CAAADRGQDTQYF   
3          TRAV16_CARPLGGSNYKLTF_TRBV2_CAAAETSNQPQHF   
4  TRAV38-2/DV8_CAYRRSASGTYKYIF_TRBV6-2_CAAAGIYNEQFF   

                     cell_id     study cancer_type          Patient  \
0    P309-GACACGCGTGGTAACG-1   Liu2025       NSCLC     Liu2025_P309   
1    P309-AGCTTGATCGACAGCC-1   Liu2025       NSCLC     Liu2025_P309   
2    P309-GGGTTGCCAAATACAG-1   Liu2025       NSCLC     Liu2025_P

In [25]:
original_study_counts = cell_info['study'].value_counts()
print("Before modification:")
print(original_study_counts)

# Modify study column: insert "_" before the 4-digit year
cell_info['study'] = cell_info['study'].str.replace(r'(\D)(\d{4})$', r'\1_\2', regex=True)

# Tabulate modified study values
modified_study_counts = cell_info['study'].value_counts()
print("\nAfter modification:")
print(modified_study_counts)

cell_info['cell_id_long'] = cell_info['cell_id'] + "-" + cell_info['study']
cell_info['cell_id_long']


Before modification:
study
Liu2025      189969
Zheng2021     70301
Chen2024      62179
Liu2022       28500
Chow2023      20061
Name: count, dtype: int64

After modification:
study
Liu_2025      189969
Zheng_2021     70301
Chen_2024      62179
Liu_2022       28500
Chow_2023      20061
Name: count, dtype: int64


0               P309-GACACGCGTGGTAACG-1-Liu_2025
1               P309-AGCTTGATCGACAGCC-1-Liu_2025
2               P309-GGGTTGCCAAATACAG-1-Liu_2025
3             P25.ut.AGAGCGAAGCTAGTTC-1-Liu_2022
4            CCCAATCTCGTCCGTT-1_PEM6C1-Chow_2023
                           ...                  
371005    CRC08-B-III_ATTACTCCATGACATC-Chen_2024
371006          P404-CATGACAGTATATCCG-1-Liu_2025
371007          P288-CACTCCACAAGCGATG-1-Liu_2025
371008       ATGGGAGTCCTTTACA-1_PEM9C5-Chow_2023
371009       GCTGCTTTCCGAGCCA.68-THCA-Zheng_2021
Name: cell_id_long, Length: 371010, dtype: object

In [26]:
adata_cells = set(adata_combined.obs_names)
meta_cells = set(cell_info['cell_id_long'])

print(list(adata_cells)[:10])
print(list(meta_cells)[:10])

# Cells in metadata but not in AnnData
missing_in_adata = meta_cells - adata_cells
# Cells in AnnData but not in metadata
missing_in_meta = adata_cells - meta_cells

print(f"\nNumber of matching cell IDs: {len(meta_cells & adata_cells)}")
print(f"Number of cell IDs in metadata not in adata_combined: {len(missing_in_adata)}")
print(f"Number of cell IDs in adata_combined not in metadata: {len(missing_in_meta)}")

ordered_meta_cells = cell_info['cell_id_long'].tolist()

adata_combined = adata_combined[ordered_meta_cells, :].copy()
assert list(adata_combined.obs_names) == ordered_meta_cells, "Ordering mismatch!"

print(f"adata_combined now contains {adata_combined.n_obs} cells.")

['CRC23-T-II_CGAGCCATCACAACGT-Chen_2024', 'AGCATACAGAGGTTAT.52-RC-Zheng_2021', 'P131-CTACCCACAAGCTGGA-1-Liu_2025', 'GCGCAGTTCGTCACGG-1_PEM2C5-Chow_2023', 'CRC19-B-II_CACCACTTCCACTGGG-Chen_2024', 'GGGCATCCATAGAAAC-1_PEM8C5-Chow_2023', 'CRC23-B-IV_ACTGAGTAGAAAGTGG-Chen_2024', 'P63-ATCATGGAGTGAAGAG-1-Liu_2025', 'P433-AGGGAGTCAGACTCGC-1-Liu_2025', 'P473-AACTCCCTCAGCACAT-1-Liu_2025']
['P377-GCGGGTTCATAGAAAC-1-Liu_2025', 'AGCATACAGAGGTTAT.52-RC-Zheng_2021', 'P131-CTACCCACAAGCTGGA-1-Liu_2025', 'CRC26-T-I_CTTACCGCATTCACTT-Chen_2024', 'GTAACTGGTTGCCTCT-1_PEM13C1-Chow_2023', 'P389-CTGAAACGTCCCTACT-1-Liu_2025', 'GCGCAGTTCGTCACGG-1_PEM2C5-Chow_2023', 'P567-CGGCTAGGTACATGTC-1-Liu_2025', 'P35.tr.1.CGGGTCAAGTCACGCC-1-Liu_2022', 'CRC23-B-IV_ACTGAGTAGAAAGTGG-Chen_2024']

Number of matching cell IDs: 371010
Number of cell IDs in metadata not in adata_combined: 0
Number of cell IDs in adata_combined not in metadata: 370454
adata_combined now contains 371010 cells.


In [27]:
cell_info = cell_info.set_index("cell_id_long")
pd.set_option("display.max_columns", None)  # Show all columns
print(cell_info.iloc[:2, :])

                                       TRB_cdr3 TRB_v_gene       TRA_cdr3  \
cell_id_long                                                                
P309-GACACGCGTGGTAACG-1-Liu_2025  CAAADRGQDTQYF     TRBV19  CAADNQAGTALIF   
P309-AGCTTGATCGACAGCC-1-Liu_2025  CAAADRGQDTQYF     TRBV19  CAADNQAGTALIF   

                                 TRA_v_gene  \
cell_id_long                                  
P309-GACACGCGTGGTAACG-1-Liu_2025   TRAV13-1   
P309-AGCTTGATCGACAGCC-1-Liu_2025   TRAV13-1   

                                                                        clone  \
cell_id_long                                                                    
P309-GACACGCGTGGTAACG-1-Liu_2025  TRAV13-1_CAADNQAGTALIF_TRBV19_CAAADRGQDTQYF   
P309-AGCTTGATCGACAGCC-1-Liu_2025  TRAV13-1_CAADNQAGTALIF_TRBV19_CAAADRGQDTQYF   

                                                  cell_id     study  \
cell_id_long                                                          
P309-GACACGCGTGGTAACG-1-Liu_2025  P

In [28]:
print(adata_combined.obs.iloc[:2, :])

                                     study  n_genes_by_counts  total_counts
P309-GACACGCGTGGTAACG-1-Liu_2025  Liu_2025               1048        2234.0
P309-AGCTTGATCGACAGCC-1-Liu_2025  Liu_2025               1116        2935.0


In [29]:
adata_combined.obs.rename(columns={"study": "study2"}, inplace=True)
adata_combined.obs = adata_combined.obs.join(cell_info)
print(adata_combined.obs.head())
adata_combined

                                        study2  n_genes_by_counts  \
P309-GACACGCGTGGTAACG-1-Liu_2025      Liu_2025               1048   
P309-AGCTTGATCGACAGCC-1-Liu_2025      Liu_2025               1116   
P309-GGGTTGCCAAATACAG-1-Liu_2025      Liu_2025               1308   
P25.ut.AGAGCGAAGCTAGTTC-1-Liu_2022    Liu_2022                967   
CCCAATCTCGTCCGTT-1_PEM6C1-Chow_2023  Chow_2023               2077   

                                     total_counts       TRB_cdr3 TRB_v_gene  \
P309-GACACGCGTGGTAACG-1-Liu_2025           2234.0  CAAADRGQDTQYF     TRBV19   
P309-AGCTTGATCGACAGCC-1-Liu_2025           2935.0  CAAADRGQDTQYF     TRBV19   
P309-GGGTTGCCAAATACAG-1-Liu_2025           2119.0  CAAADRGQDTQYF     TRBV19   
P25.ut.AGAGCGAAGCTAGTTC-1-Liu_2022         1585.0  CAAAETSNQPQHF      TRBV2   
CCCAATCTCGTCCGTT-1_PEM6C1-Chow_2023        7606.0   CAAAGIYNEQFF    TRBV6-2   

                                            TRA_cdr3    TRA_v_gene  \
P309-GACACGCGTGGTAACG-1-Liu_2025       C

AnnData object with n_obs × n_vars = 371010 × 6256
    obs: 'study2', 'n_genes_by_counts', 'total_counts', 'TRB_cdr3', 'TRB_v_gene', 'TRA_cdr3', 'TRA_v_gene', 'clone', 'cell_id', 'study', 'cancer_type', 'Patient', 'Sample', 'Treatment', 'Tissue', 'study_specific_CR_per_cell', 'study_specific_CR_by_cluster', 'barcode', 'TRA_j_gene', 'TRB_j_gene', 'TRA_nUMI', 'TRB_nUMI', 'CD8_Lowery_pos_243g', 'CD8_Oliveira_TTE_100g', 'CD8_Oliveira_pos_74g', 'CD8_Yost_CD8_Exh_100g', 'CD8_ave_Hanada_pos_27g', 'CD8_ave_Hanada_neg_5g', 'CD8_ave_Oliveira_virus_26g', 'study_clone_id', 'study_clone_number', 'pos_score_CD8', 'neg_score_CD8', 'cancer_reactive_per_cell', 'cancer_reactive', 'total_cells_patient', 'clone_number_per_patient', 'clone_number_total', 'clone_n_patient', 'clone_number_per_patient_median', 'clone_freq_per_patient', 'TRA_antigen', 'TRB_antigen', 'non_human_antigen', 'label'
    var: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [30]:
adata_combined.write(os.path.join(meta_dir, "CD8_combined_filtered.h5ad"))

print(f"Saved filtered AnnData to: {os.path.join(meta_dir, 'CD8_combined_filtered.h5ad')}")


Saved filtered AnnData to: /Users/wsun/research/CAT/data/CD8_combined_filtered.h5ad
